# 06 - CTD chemical-gene-disease reasoning with LoRA SFT

This notebook builds a small evidence-grounded reasoning dataset from CTD chemical-gene interactions plus the **curated** CTD gene-disease associations. It does not use the multi-gigabyte aggregate gene-disease file.

Pipeline: CTD chemical-gene + curated gene-disease -> 2-hop evidence chains -> chemical-disjoint split -> Base evaluation -> LoRA SFT -> After evaluation.

The goal is to test whether a small instruct model can learn to produce short, evidence-grounded chemical -> gene -> disease reasoning chains.

In [ ]:
!pip -q install -U "transformers>=4.55,<5" "datasets>=3.6,<5" "peft>=0.17,<1" "trl==0.29.1" "accelerate>=1.10,<2" "bitsandbytes>=0.46,<1" "torchao>=0.16,<1"


In [ ]:
import os
import re
import ast
import random
import pandas as pd
import torch
from datasets import Dataset
from google.colab import files

import transformers, datasets, peft, trl
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('Datasets:', datasets.__version__)
print('PEFT:', peft.__version__)
print('TRL:', trl.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

print('Upload these two files:')
print('  1) CTD_chem_gene_ixns.tsv.gz')
print('  2) CTD_curated_genes_diseases.tsv.gz')
uploaded = files.upload()

CHEM_GENE = '/content/CTD_chem_gene_ixns.tsv.gz'
GENE_DISEASE = '/content/CTD_curated_genes_diseases.tsv.gz'
if not os.path.exists(CHEM_GENE):
    raise FileNotFoundError('Missing CTD_chem_gene_ixns.tsv.gz')
if not os.path.exists(GENE_DISEASE):
    raise FileNotFoundError('Missing CTD_curated_genes_diseases.tsv.gz')

chem_cols = ['ChemicalName','ChemicalID','CasRN','GeneSymbol','GeneID','GeneForms','Organism','OrganismID','Interaction','InteractionActions','PubMedIDs']
gd_cols = ['GeneSymbol','GeneID','DiseaseName','DiseaseID','DirectEvidence','InferenceChemicalName','InferenceChemicalID','OmimIDs','PubMedIDs']
chem = pd.read_csv(CHEM_GENE, sep='\t', comment='#', header=None, names=chem_cols, dtype=str, low_memory=False)
gd = pd.read_csv(GENE_DISEASE, sep='\t', comment='#', header=None, names=gd_cols, dtype=str, low_memory=False)

chem = chem[chem['OrganismID'].fillna('').str.strip().eq('9606')].copy()
chem = chem[chem['ChemicalName'].notna() & chem['GeneSymbol'].notna() & chem['GeneID'].notna()].copy()
gd = gd[gd['GeneID'].notna() & gd['DiseaseName'].notna() & gd['DiseaseID'].notna()].copy()

chem['GeneID'] = chem['GeneID'].str.replace(r'\.0$', '', regex=True)
gd['GeneID'] = gd['GeneID'].str.replace(r'\.0$', '', regex=True)

# Keep curated gene-disease rows only; the curated file itself is the intended source.
# Deduplicate to one evidence row per gene-disease pair.
gd = gd.drop_duplicates(['GeneID','DiseaseID']).copy()

# Join CTD chemical-gene evidence to curated gene-disease evidence on GeneID.
pairs = chem.merge(gd[['GeneID','DiseaseName','DiseaseID','DirectEvidence']], on='GeneID', how='inner')
pairs = pairs[pairs['ChemicalName'].notna() & pairs['GeneSymbol'].notna() & pairs['DiseaseName'].notna()].copy()
pairs = pairs.drop_duplicates(['ChemicalID','GeneID','DiseaseID']).reset_index(drop=True)

print('Chemical-gene rows after human filter:', len(chem))
print('Curated gene-disease rows:', len(gd))
print('Two-hop chemical-gene-disease paths:', len(pairs))
print(pairs[['ChemicalName','GeneSymbol','DiseaseName','InteractionActions']].head())


In [ ]:
# Build short evidence-grounded reasoning examples.
MAX_EXAMPLES = 5000
pairs = pairs.sample(frac=1.0, random_state=42).reset_index(drop=True)

records = []
seen = set()
for _, r in pairs.iterrows():
    key = (str(r['ChemicalID']), str(r['GeneID']), str(r['DiseaseID']))
    if key in seen:
        continue
    seen.add(key)
    action = str(r['InteractionActions']).strip()
    if action in ('nan', ''):
        action = str(r['Interaction']).strip()
    chemical = str(r['ChemicalName']).strip()
    gene = str(r['GeneSymbol']).strip()
    disease = str(r['DiseaseName']).strip()
    prompt = (
        f'Evidence 1: CTD reports a chemical-gene relationship for {chemical} and gene {gene}'
        + (f' with interaction action: {action}.' if action else '.')
        + f'\nEvidence 2: CTD curator data links gene {gene} to disease {disease}.'
        + f'\nQuestion: What disease is connected to {chemical} through gene {gene}? '
          'Give the disease name and a concise two-hop reasoning chain.'
    )
    answer = (
        f'Disease: {disease}. Reasoning: {chemical} -> {gene} -> {disease}. '
        f'The first hop is supported by the CTD chemical-gene relationship, and the second hop is supported by the curated CTD gene-disease association.'
    )
    records.append({
        'chemical': chemical,
        'gene': gene,
        'disease': disease,
        'chemical_id': str(r['ChemicalID']),
        'gene_id': str(r['GeneID']),
        'disease_id': str(r['DiseaseID']),
        'prompt': prompt,
        'answer': answer,
    })
    if len(records) >= MAX_EXAMPLES:
        break

data = pd.DataFrame(records)
print('Examples:', len(data))
print(data[['chemical','gene','disease']].head())

# Chemical-disjoint split: evaluation chemicals are unseen during training.
chemicals = data['chemical_id'].drop_duplicates().sample(frac=1.0, random_state=42).tolist()
cut = max(1, int(len(chemicals) * 0.1))
eval_chems = set(chemicals[:cut])
train_df = data[~data['chemical_id'].isin(eval_chems)].copy()
eval_df = data[data['chemical_id'].isin(eval_chems)].copy()

# Keep evaluation compact for Colab generation.
eval_df = eval_df.head(300).copy()
train_ds = Dataset.from_pandas(train_df[['prompt','answer']], preserve_index=False)
eval_ds = Dataset.from_pandas(eval_df[['prompt','answer','chemical','gene','disease']], preserve_index=False)
print('Train:', len(train_ds), 'Eval:', len(eval_ds), 'Eval unique chemicals:', eval_df['chemical_id'].nunique())


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

def make_chat_text(prompt, answer):
    messages = [
        {'role': 'user', 'content': prompt},
        {'role': 'assistant', 'content': answer},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

train_ds = train_ds.map(lambda x: {'text': make_chat_text(x['prompt'], x['answer'])})
eval_ds = eval_ds.map(lambda x: {'text': make_chat_text(x['prompt'], x['answer'])})


In [ ]:
# Before-SFT generation evaluation.
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype).cuda()
model.eval()

def normalize(s):
    return re.sub(r'[^a-z0-9]+', ' ', str(s).lower()).strip()

def generate_batch(model, prompts, max_new_tokens=96, batch_size=8):
    outputs = []
    for start in range(0, len(prompts), batch_size):
        batch = prompts[start:start+batch_size]
        texts = [
            tokenizer.apply_chat_template([{'role':'user','content': p}], tokenize=False, add_generation_prompt=True)
            for p in batch
        ]
        enc = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=384)
        enc = {k: v.to(model.device) for k,v in enc.items()}
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        n = enc['input_ids'].shape[1]
        gen = out[:, n:]
        outputs.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
    return outputs

def score_reasoning(df, predictions):
    disease_hits = []
    chain_hits = []
    for (_, row), pred in zip(df.iterrows(), predictions):
        p = normalize(pred)
        disease_ok = normalize(row['disease']) in p
        gene_ok = normalize(row['gene']) in p
        disease_hits.append(disease_ok)
        chain_hits.append(disease_ok and gene_ok)
    return {
        'disease_accuracy': sum(disease_hits)/len(disease_hits),
        'chain_accuracy': sum(chain_hits)/len(chain_hits),
    }

before_predictions = generate_batch(model, eval_df['prompt'].tolist())
before_metrics = score_reasoning(eval_df, before_predictions)
print('BEFORE SFT')
print(before_metrics)
for i in range(min(3, len(eval_df))):
    print('='*80)
    print(eval_df.iloc[i]['prompt'])
    print('PRED:', before_predictions[i])
    print('GOLD:', eval_df.iloc[i]['answer'])


In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)

sft_args = SFTConfig(
    output_dir='./outputs/ctd-reasoning-sft',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    max_length=512,
    logging_steps=20,
    save_strategy='no',
    report_to='none',
    gradient_checkpointing=False,
    use_cache=False,
    packing=True,
    dataset_text_field='text',
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.train()


In [ ]:
model.eval()
after_predictions = generate_batch(model, eval_df['prompt'].tolist())
after_metrics = score_reasoning(eval_df, after_predictions)

print('AFTER SFT')
print(after_metrics)
print('')
print('METRIC COMPARISON')
print('-'*72)
print(f"{'Metric':<28}{'Before':>12}{'After':>12}{'Delta':>12}")
print('-'*72)
for key in ['disease_accuracy','chain_accuracy']:
    b = before_metrics[key]
    a = after_metrics[key]
    print(f"{key:<28}{b:>12.3f}{a:>12.3f}{a-b:>12.3f}")

for i in range(min(5, len(eval_df))):
    print('='*80)
    print('CHEMICAL:', eval_df.iloc[i]['chemical'])
    print('GENE:', eval_df.iloc[i]['gene'])
    print('GOLD DISEASE:', eval_df.iloc[i]['disease'])
    print('BEFORE:', before_predictions[i])
    print('AFTER :', after_predictions[i])


In [ ]:
ADAPTER_DIR = './outputs/ctd-reasoning-sft-adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Saved:', ADAPTER_DIR)


## Interpretation

This experiment tests **reasoning from explicit evidence**, not biomedical knowledge discovery. The model is trained to map two CTD-supported facts into a short Chemical -> Gene -> Disease chain.

The evaluation is chemical-disjoint, so held-out chemicals are unseen during training. It is still a small teaching experiment: disease/gene overlap across train and test can occur, and substring matching is only a coarse metric.

The multi-gigabyte aggregate `CTD_genes_diseases.tsv.gz` is intentionally not used. Use the much smaller curated `CTD_curated_genes_diseases.tsv.gz` together with `CTD_chem_gene_ixns.tsv.gz`. CTD tooling documentation identifies the curated gene-disease file as the manually curated subset of the larger gene-disease file.
